## Silver Layer - Public Holidays

This notebook transforms bronze layer holidays data into a clean, standardized format for analytics.

**Transformations:**
- Normalize country codes (GB → UK for consistency with sales data)
- Add computed columns for holiday analysis
- Filter to date range matching sales data (2010-12-29 to 2014-01-28)
- Add day of week and proximity flags for analysis

**Output:** `silver.holidays`

In [0]:
%sql
CREATE OR REPLACE TABLE silver.holidays AS
SELECT
  -- Normalize country code: GB -> UK to match sales data
  CASE 
    WHEN country_code = 'GB' THEN 'UK'
    ELSE country_code
  END AS country_code,
  
  country_name,
  date AS holiday_date,
  holiday_name,
  local_name,
  is_global,
  
  -- Add computed columns for analysis
  YEAR(date) AS year,
  MONTH(date) AS month,
  DAYOFWEEK(date) AS day_of_week,
  DATE_FORMAT(date, 'EEEE') AS day_name,
  
  -- Major holiday flags for analysis
  CASE
    WHEN holiday_name LIKE '%Christmas%' THEN TRUE
    WHEN holiday_name LIKE '%New Year%' THEN TRUE
    WHEN holiday_name LIKE '%Thanksgiving%' THEN TRUE
    WHEN holiday_name LIKE '%Easter%' THEN TRUE
    ELSE FALSE
  END AS is_major_holiday,
  
  ingestion_timestamp
  
FROM bronze.holidays_api_raw
WHERE date BETWEEN '2010-12-29' AND '2014-01-28'  -- Match sales data range
ORDER BY date, country_code

In [0]:
%sql
-- Validation: Check holidays by country
SELECT 
  country_code,
  country_name,
  COUNT(*) AS total_holidays,
  COUNT(DISTINCT year) AS years_covered,
  SUM(CASE WHEN is_major_holiday THEN 1 ELSE 0 END) AS major_holidays
FROM silver.holidays
GROUP BY country_code, country_name
ORDER BY country_code